In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pdfplumber
import nltk
import re

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\ishit\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ishit\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7821.51it/s]
BertMo

In [5]:
def extract_text_from_pdf(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [6]:
def clean_text(text):
    text =  text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text

In [7]:
def get_embedding(text):
    return model.encode(text)

In [8]:
def calculate_similarity(resume_text, jd_text):
    resume_embedding = get_embedding(resume_text)
    jd_embedding = get_embedding(jd_text)

    score = cosine_similarity([resume_embedding], [jd_embedding])[0][0]
    return round(score * 100, 2)

In [10]:
SKILLS = [
    "python", "java", "sql", "react", "node",
    "machine learning", "deep learning",
    "tableau", "power bi", "excel",
    "aws", "azure", "docker"
]

In [11]:
def extract_skills(text):
    found = []
    for skill in SKILLS:
        if skill in text:
            found.append(skill)
    return set(found)

In [12]:
def compare_skills(resume_text, jd_text):
    resume_skills = extract_skills(resume_text)
    jd_skills = extract_skills(jd_text)

    matched = list(resume_skills & jd_skills)
    missing = list(jd_skills - resume_skills)

    return matched, missing

In [14]:
resume_path = r"D:\Resume\Ishita Sinha.pdf"
jd_text = """We are looking for a Python developer with exexperience in SQL, AWS, and Machine Learning."""

resume_text = extract_text_from_pdf(resume_path)

resume_text = clean_text(resume_text)
jd_text = clean_text(jd_text)

score = calculate_similarity(resume_text, jd_text)
matched, missing = compare_skills(resume_text, jd_text)

print("Match Score:", score, "%")
print("Matched Skills:", matched)
print("Missing Skills:", missing)

Match Score: 37.48 %
Matched Skills: ['python', 'sql', 'machine learning']
Missing Skills: ['aws']
